In [ ]:
%load_ext autoreload
%autoreload 2
import os
import torch
import numpy as np
import json



from compactreasoningmodels.datasets import NonogramDataset
from compactreasoningmodels.datasets.collate import collate_raw

if 'original_dir' not in globals():
    original_dir = os.getcwd()

os.chdir(os.path.join(original_dir, ".."))
os.environ["DATA_DIR"] = os.path.join(os.getcwd(), "data")
os.environ["MODEL_DIR"] = os.path.join(os.getcwd(), "models")


In [ ]:
dataset = NonogramDataset("traces/nonograms2_5x5.jsonl")
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_raw)

In [ ]:
import pandas as pd
from itertools import chain

puzzle_idx_counter = 0
puzzle_lookup = {}
runs_list = []
steps_list = []

for X, y, meta in chain.from_iterable(
    zip(X_batch, y_batch, meta_batch)
    for X_batch, y_batch, meta_batch in dataloader
):
    puzzle_idx = puzzle_idx_counter
    puzzle_idx_counter += 1
    puzzle_lookup[puzzle_idx] = (X, y)

    traces = meta["traces"]
    shape = meta["shape"]
    density = meta["density"]
    mean_clue_runs = meta["mean_clue_runs"]

    for solver_name, solver in traces.items():
        for sr_name, sr in solver.items():
            sampling_ratio = float(sr_name)
            for i, trace in enumerate(sr):
                run_idx = len(runs_list)  # global synthetic run id

                runs_list.append({
                    'run_idx': run_idx,
                    'puzzle_idx': puzzle_idx,
                    'shape': shape,
                    'density': density,
                    'mean_clue_runs': mean_clue_runs,
                    'solver': solver_name,
                    'sampling_ratio': sampling_ratio,
                    'run_index': i,
                    'solved': trace["solved"],
                    'num_steps': trace["num_steps"],
                    'step_ratio': trace["step_ratio"],
                    'mse_loss_final': trace["mse_losses"][-1] if trace["mse_losses"] else None,
                    'cr_loss_final': trace["cr_losses"][-1] if trace["cr_losses"] else None,
                    'steps_to_solve': trace["steps_to_solve"],
                })

                for step, cr, mse in zip(trace["steps"], trace["cr_losses"], trace["mse_losses"]):
                    steps_list.append({
                        'run_idx': run_idx,
                        'step': step,
                        'mse_loss': mse,
                        'cr_loss': cr,
                    })

df_runs = pd.DataFrame(runs_list)
df_steps = pd.DataFrame(steps_list)

In [ ]:
summary = df_runs.groupby(['solver', 'sampling_ratio']).agg(
    max_steps_to_solve=('steps_to_solve', 'max'),
    mean_steps_to_solve=('steps_to_solve', lambda x: x[x != -1].mean()),
)
print(summary)

In [ ]:
summary = (
    df_runs.groupby(["solver", "sampling_ratio"])
      .agg(
          accuracy_mean=("solved", "mean"), accuracy_std=("solved", "std"),
          mse_mean=("mse_loss_final", "mean"), mse_std=("mse_loss_final", "std"),
          cr_mean=("cr_loss_final", "mean"), cr_std=("cr_loss_final", "std"),
          n_runs=("run_idx", "count"),
      )
      .reset_index()
)
print(summary)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt

# --- 1. Pivot: one row per (solver, sampling_ratio) config, one column per puzzle ---
pivot = df_runs.pivot_table(
    index=["solver", "sampling_ratio"],
    columns="puzzle_idx",
    values="mse_loss_final",
    aggfunc="mean",
)

print(f"Shape: {pivot.shape[0]} configs x {pivot.shape[1]} puzzles")
print(f"Missing values: {pivot.isna().sum().sum()} of {pivot.size}")

# --- 2. Handle missing puzzle coverage + standardize ---
# Standardizing matters here: puzzles vary wildly in difficulty, so without
# scaling, PC1 would mostly reflect "which puzzles are hardest" rather than
# "how configs differ from each other" — same effect but on the puzzle axis now.
imputer = SimpleImputer(strategy="mean")
X = imputer.fit_transform(pivot.values)
X = StandardScaler().fit_transform(X)

# --- 3. PCA to 2D ---
pca = PCA(n_components=2)
coords = pca.fit_transform(X)

print(f"Explained variance: PC1={pca.explained_variance_ratio_[0]:.1%}, "
      f"PC2={pca.explained_variance_ratio_[1]:.1%}")

result = pivot.index.to_frame(index=False)
result["pc1"] = coords[:, 0]
result["pc2"] = coords[:, 1]

# --- 4. Plot: each point = one (solver, sampling_ratio) config ---
fig, ax = plt.subplots(figsize=(8, 6))
solvers = result["solver"].unique()
cmap = plt.get_cmap("tab10")

for i, solver in enumerate(solvers):
    subset = result[result.solver == solver]
    ax.scatter(subset["pc1"], subset["pc2"], color=cmap(i % 10), s=80, label=solver)
    for _, row in subset.iterrows():
        ax.annotate(f"sr={row['sampling_ratio']}", (row["pc1"], row["pc2"]),
                    fontsize=8, xytext=(4, 4), textcoords="offset points")

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
ax.set_title("PCA of (solver, sampling_ratio) configs across puzzles, by MSE")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from itertools import chain
from collections import defaultdict

N_BUCKETS = 10
bucket_edges = np.linspace(0, 1, N_BUCKETS + 1)

# accumulator: (solver, sampling_ratio) -> dict of step_idx -> length-10 count array
dist_accum = defaultdict(lambda: defaultdict(lambda: np.zeros(N_BUCKETS, dtype=np.int64)))

for X, y, meta in chain.from_iterable(
    zip(X_batch, y_batch, meta_batch)
    for X_batch, y_batch, meta_batch in dataloader
):
    traces = meta["traces"]

    for solver_name, solver in traces.items():
        for sr_name, sr in solver.items():
            sampling_ratio = float(sr_name)
            key = (solver_name, sampling_ratio)

            for trace in sr:  # each trace = one run
                for step_idx, grid in enumerate(trace["steps"]):
                    grid_arr = np.asarray(grid)
                    counts, _ = np.histogram(grid_arr.ravel(), bins=bucket_edges)
                    dist_accum[key][step_idx] += counts

# --- Stack into a 2D array per (solver, sampling_ratio): shape (num_steps, 10) ---
dist_matrices = {}
for key, step_dict in dist_accum.items():
    max_step = max(step_dict.keys())
    matrix = np.zeros((max_step + 1, N_BUCKETS), dtype=np.int64)
    for step_idx, counts in step_dict.items():
        matrix[step_idx] = counts
    dist_matrices[key] = matrix

# Example: inspect one config
solver, sampling_ratio = list(dist_matrices.keys())[0]
print(f"{solver}, sr={sampling_ratio}: shape {dist_matrices[(solver, sampling_ratio)].shape}")
print(dist_matrices[(solver, sampling_ratio)])

In [ ]:
import matplotlib.pyplot as plt

def plot_distribution_heatmap(matrix, title, normalize=True):
    if normalize:
        # normalize each step's row to sum to 1 (since later steps have fewer contributing runs)
        row_sums = matrix.sum(axis=1, keepdims=True)
        matrix = np.divide(matrix, row_sums, out=np.zeros_like(matrix, dtype=float), where=row_sums != 0)

    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(matrix.T, aspect="auto", origin="lower", cmap="viridis",
                    extent=[0, matrix.shape[0], 0, 1])
    ax.set_xlabel("Step")
    ax.set_ylabel("Cell value bucket (0-1)")
    ax.set_title(title)
    plt.colorbar(im, label="proportion of cells" if normalize else "cell count")
    plt.tight_layout()
    plt.show()

for (solver, sampling_ratio), matrix in dist_matrices.items():
    plot_distribution_heatmap(matrix, f"{solver}, sampling_ratio={sampling_ratio}")

In [ ]:
from collections import defaultdict
import numpy as np
import pandas as pd

results = defaultdict(lambda: {"accuracy": [], "mse_loss": [], "cr_loss": []})

for clues, grid, meta in dataloader:
    true_grid = grid[0]
    traces = meta[0].get("traces", None)
    if not traces:
        continue

    for trace_name, sampling_ratios in traces.items():
        for sampling_ratio, runs in sampling_ratios.items():
            key = (trace_name, sampling_ratio)
            results[key]["accuracy"].append(np.mean([run["solved"] for run in runs]))
            results[key]["mse_loss"].append(np.mean([run["mse_losses"][-1] for run in runs]))
            results[key]["cr_loss"].append(np.mean([run["cr_losses"][-1] for run in runs]))

# Build a tidy summary table: mean ± std for each metric
rows = []
for (trace_name, sampling_ratio), metrics in results.items():
    row = {"trace": trace_name, "sampling_ratio": sampling_ratio}
    for metric_name, values in metrics.items():
        row[f"{metric_name}_mean"] = np.mean(values)
        row[f"{metric_name}_std"] = np.std(values)
    rows.append(row)

df = pd.DataFrame(rows).sort_values(["trace", "sampling_ratio"]).reset_index(drop=True)
df

In [ ]:
from collections import defaultdict
from itertools import product
import numpy as np
import pandas as pd

# metric_name -> (run_key, how to extract the value from run[run_key])
metric_extractors = {
    "accuracy": ("solved", lambda v: v),              # scalar per run
    "mse_loss": ("mse_losses", lambda v: v[-1]),       # last-step loss
    "cr_loss": ("cr_losses", lambda v: v[-1]),         # last-step loss
}
metrics = list(metric_extractors.keys())

data = defaultdict(lambda: defaultdict(list))

for clues, grid, meta in dataloader:
    traces = meta[0].get("traces", None)
    if not traces:
        continue
    for trace_name, sampling_ratios in traces.items():
        for sampling_ratio, runs in sampling_ratios.items():
            if not runs:
                continue
            key = (trace_name, sampling_ratio)
            for metric_name, (run_key, extract) in metric_extractors.items():
                run_values = [extract(run[run_key]) for run in runs]
                data[key][metric_name].append(run_values)

keys = sorted(data.keys())
key_labels = {k: f"{k[0]}|{k[1]}" for k in keys}

def paired_values(key_a, key_b, metric):
    a_vals, b_vals = [], []
    for runs_a, runs_b in zip(data[key_a][metric], data[key_b][metric]):
        for i, va in enumerate(runs_a):
            for j, vb in enumerate(runs_b):
                if key_a == key_b and i == j:
                    continue
                a_vals.append(va)
                b_vals.append(vb)
    return np.array(a_vals), np.array(b_vals)

labels = [key_labels[k] for k in keys]
corr_matrices = {
    metric: pd.DataFrame(index=labels, columns=labels, dtype=float)
    for metric in metrics
}

for metric in metrics:
    for k1, k2 in product(keys, keys):
        a, b = paired_values(k1, k2, metric)
        if len(a) < 2 or np.std(a) == 0 or np.std(b) == 0:
            val = np.nan
        else:
            val = np.corrcoef(a, b)[0, 1]
        corr_matrices[metric].loc[key_labels[k1], key_labels[k2]] = val

for metric in metrics:
    print(f"Correlation matrix ({metric}):")
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        display(corr_matrices[metric])

In [ ]:
from itertools import combinations
# create new dataset with puzzle index and mse, filter to only do sampling ratio = 1.0
df_filtered = df_runs.copy()
df_filtered = df_runs[df_runs["sampling_ratio"] == 1.0]
df_filtered = df_filtered[df_filtered["mse_loss_final"] != 0]

# Step 1: Get top 10 puzzle_idx per solver as sets
puzzle_sets = (df_filtered.sort_values('mse_loss_final', ascending=False)
               .groupby('solver', sort=False)
               .head(10)
               .groupby('solver')['puzzle_idx']
               .apply(set)
               .to_dict())
solver_names = sorted(puzzle_sets.keys())
jaccard_matrix = pd.DataFrame(index=solver_names, columns=solver_names)

for s1 in solver_names:
    for s2 in solver_names:
        if s1 == s2:
            jaccard_matrix.loc[s1, s2] = 1.0
        else:
            union = puzzle_sets[s1] | puzzle_sets[s2]
            jaccard_matrix.loc[s1, s2] = len(puzzle_sets[s1] & puzzle_sets[s2]) / len(union)
jaccard_matrix